In [1]:
import pandas as pd
xls = pd.ExcelFile("../data/raw/Supply chain logistics problem.xlsx", engine="openpyxl")
print(xls.sheet_names)


['OrderList', 'FreightRates', 'WhCosts', 'WhCapacities', 'ProductsPerPlant', 'VmiCustomers', 'PlantPorts']


In [28]:
wh_costs = pd.read_excel(xls, sheet_name = "WhCosts")
print(wh_costs.head())

wh_capacities = pd.read_excel(xls, sheet_name = "WhCapacities")
print(wh_capacities.head())

freight_rates = pd.read_excel(xls, sheet_name = "FreightRates")
print(freight_rates.head())

plant_ports = pd.read_excel(xls, sheet_name = "PlantPorts")
print(plant_ports.head())

        WH  Cost/unit
0  PLANT15   1.415063
1  PLANT17   0.428947
2  PLANT18   2.036254
3  PLANT05   0.488144
4  PLANT02   0.477504
  Plant ID  Daily Capacity 
0  PLANT15               11
1  PLANT17                8
2  PLANT18              111
3  PLANT05              385
4  PLANT02              138
  Carrier orig_port_cd dest_port_cd  minm_wgh_qty  max_wgh_qty svc_cd  \
0  V444_6       PORT08       PORT09         250.0       499.99    DTD   
1  V444_6       PORT08       PORT09          65.0        69.99    DTD   
2  V444_6       PORT08       PORT09          60.0        64.99    DTD   
3  V444_6       PORT08       PORT09          50.0        54.99    DTD   
4  V444_6       PORT08       PORT09          35.0        39.99    DTD   

   minimum cost    rate mode_dsc  tpt_day_cnt Carrier type  
0       43.2272  0.7132   AIR               2  V88888888_0  
1       43.2272  0.7512   AIR               2  V88888888_0  
2       43.2272  0.7892   AIR               2  V88888888_0  
3       43.2272  

In [29]:
order_list = pd.read_excel(xls, sheet_name = "OrderList")
print(order_list.head())
print(order_list.shape)

demand_by_port = order_list.groupby("Destination Port")["Unit quantity"].sum().reset_index()
print(demand_by_port)

       Order ID Order Date Origin Port Carrier  TPT Service Level  \
0  1.447296e+09 2013-05-26      PORT09   V44_3    1           CRF   
1  1.447158e+09 2013-05-26      PORT09   V44_3    1           CRF   
2  1.447139e+09 2013-05-26      PORT09   V44_3    1           CRF   
3  1.447364e+09 2013-05-26      PORT09   V44_3    1           CRF   
4  1.447364e+09 2013-05-26      PORT09   V44_3    1           CRF   

   Ship ahead day count  Ship Late Day count   Customer  Product ID  \
0                     3                    0  V55555_53     1700106   
1                     3                    0  V55555_53     1700106   
2                     3                    0  V55555_53     1700106   
3                     3                    0  V55555_53     1700106   
4                     3                    0  V55555_53     1700106   

  Plant Code Destination Port  Unit quantity  Weight  
0    PLANT16           PORT09            808   14.30  
1    PLANT16           PORT09           3188   8

In [30]:
print(order_list["Customer"].nunique())
demand_by_customer = order_list.groupby("Customer")["Unit quantity"].sum().reset_index()
print(demand_by_customer.head(10))

46
                 Customer  Unit quantity
0  V555555555555555555_17         266457
1  V555555555555555555_42         470632
2  V555555555555555555_45         116136
3  V555555555555555555_46          12080
4     V555555555555555_23            375
5     V555555555555555_29        1054980
6     V555555555555555_44          15125
7       V55555555555555_8         877824
8       V5555555555555_16           3434
9        V555555555555_31         435868


In [31]:
merged = order_list.merge(freight_rates, left_on = ["Carrier", "Origin Port", "Destination Port"], right_on = ["Carrier", "orig_port_cd", "dest_port_cd"], how = "left")
print(merged.head())
print(merged.shape)

       Order ID Order Date Origin Port Carrier  TPT Service Level  \
0  1.447296e+09 2013-05-26      PORT09   V44_3    1           CRF   
1  1.447158e+09 2013-05-26      PORT09   V44_3    1           CRF   
2  1.447139e+09 2013-05-26      PORT09   V44_3    1           CRF   
3  1.447364e+09 2013-05-26      PORT09   V44_3    1           CRF   
4  1.447364e+09 2013-05-26      PORT09   V44_3    1           CRF   

   Ship ahead day count  Ship Late Day count   Customer  Product ID  ...  \
0                     3                    0  V55555_53     1700106  ...   
1                     3                    0  V55555_53     1700106  ...   
2                     3                    0  V55555_53     1700106  ...   
3                     3                    0  V55555_53     1700106  ...   
4                     3                    0  V55555_53     1700106  ...   

  orig_port_cd dest_port_cd  minm_wgh_qty  max_wgh_qty svc_cd minimum cost  \
0          NaN          NaN           NaN         

In [32]:
print(order_list["Carrier"].unique()[:10])
print(freight_rates["Carrier"].unique()[:10])

print(order_list["Origin Port"].unique()[:10])
print(freight_rates["orig_port_cd"].unique()[:10])



<StringArray>
['V44_3', 'V444_0', 'V444_1']
Length: 3, dtype: str
<StringArray>
['V444_6', 'V444_8', 'V444_9', 'V444_2', 'V444_1', 'V444_0', 'V444_5',
 'V444_4', 'V444_7']
Length: 9, dtype: str
<StringArray>
['PORT09', 'PORT04', 'PORT05']
Length: 3, dtype: str
<StringArray>
['PORT08', 'PORT10', 'PORT09', 'PORT11', 'PORT04', 'PORT02', 'PORT03',
 'PORT07', 'PORT05', 'PORT06']
Length: 10, dtype: str


In [33]:
matching_carriers = order_list["Carrier"].isin(["V444_0", "V444_1"])
print(matching_carriers.sum())

8361


In [34]:
merged = order_list.merge(freight_rates, left_on = ["Carrier", "Origin Port", "Destination Port"], right_on = ["Carrier", "orig_port_cd", "dest_port_cd"], how = "inner")
print(merged.shape)

final = merged[(merged["Weight"] >= merged["minm_wgh_qty"]) & (merged["Weight"] <= merged["max_wgh_qty"])]
print(final.shape)

final_sorted = final.sort_values("rate")
final_dedup =  final_sorted.drop_duplicates(subset = "Order ID", keep = "first")
print(final_dedup.shape)
print(final.shape)

(208548, 24)
(27896, 24)
(6991, 24)
(27896, 24)


In [35]:
plant_customer_cost = final_dedup.groupby(["Plant Code", "Customer"])["rate"].mean().reset_index()
print(plant_customer_cost.head(10))
print(plant_customer_cost.shape)

  Plant Code                Customer      rate
0    PLANT03  V555555555555555555_17  0.293046
1    PLANT03  V555555555555555555_42  0.048400
2    PLANT03  V555555555555555555_45  0.237600
3    PLANT03  V555555555555555555_46  0.299600
4    PLANT03     V555555555555555_29  0.048319
5    PLANT03     V555555555555555_44  0.080000
6    PLANT03       V55555555555555_8  0.296215
7    PLANT03       V5555555555555_16  0.234182
8    PLANT03         V55555555555_28  0.048341
9    PLANT03           V555555555_14  0.048356
(68, 3)


In [36]:
demand = final_dedup.groupby("Customer")["Unit quantity"].sum().reset_index()
print(demand.head(10))
print(demand.shape)

                 Customer  Unit quantity
0  V555555555555555555_17          28995
1  V555555555555555555_42         470632
2  V555555555555555555_45          62745
3  V555555555555555555_46          12080
4     V555555555555555_29         102573
5     V555555555555555_44           9333
6       V55555555555555_8         435751
7       V5555555555555_16           3170
8         V55555555555_28        5293676
9           V5555555555_1           6441
(42, 2)


In [37]:
print(wh_capacities.head())
print(wh_capacities["Plant ID"].nunique())
print(plant_customer_cost["Plant Code"].nunique())

  Plant ID  Daily Capacity 
0  PLANT15               11
1  PLANT17                8
2  PLANT18              111
3  PLANT05              385
4  PLANT02              138
19
6


In [38]:
plants_in_cost = set(plant_customer_cost["Plant Code"].unique())
plants_in_capacity =  set(wh_capacities["Plant ID"].unique())

print(plants_in_cost & plants_in_capacity)
print(plants_in_capacity)
print(plants_in_cost)

{'PLANT13', 'PLANT16', 'PLANT09', 'PLANT03', 'PLANT12', 'PLANT08'}
{'PLANT16', 'PLANT05', 'PLANT03', 'PLANT12', 'PLANT18', 'PLANT01', 'PLANT13', 'PLANT17', 'PLANT11', 'PLANT07', 'PLANT10', 'PLANT14', 'PLANT02', 'PLANT19', 'PLANT08', 'PLANT04', 'PLANT09', 'PLANT15', 'PLANT06'}
{'PLANT13', 'PLANT16', 'PLANT09', 'PLANT03', 'PLANT12', 'PLANT08'}


In [39]:
print(order_list["Order Date"].min())
print(order_list["Order Date"].max())

days = (order_list["Order Date"].max() - order_list["Order Date"].min()).days
print(days)

2013-05-26 00:00:00
2013-05-26 00:00:00
0


In [40]:
total_demand = demand["Unit quantity"].sum()
total_capacity_available = wh_capacities[wh_capacities["Plant ID"].isin(plants_in_cost)]["Daily Capacity "].sum()

print("Total demand:", total_demand)
print("Total available capacity:", total_capacity_available)

Total demand: 24294442
Total available capacity: 2194


In [41]:
print(order_list["Carrier"].value_counts())

Carrier
V444_0    6264
V444_1    2097
V44_3      854
Name: count, dtype: int64


In [42]:
wh_capacities.columns = wh_capacities.columns.str.strip()

wh_subset = wh_capacities[wh_capacities["Plant ID"].isin(plants_in_cost)].copy()

wh_subset["capacity_share"] = wh_subset["Daily Capacity"] / wh_subset["Daily Capacity"].sum()

wh_subset["scaled_capacity"] = wh_subset["capacity_share"] * total_demand * 1.15

print(wh_subset)

   Plant ID  Daily Capacity  capacity_share  scaled_capacity
10  PLANT16             457        0.208295     5.819482e+06
11  PLANT12             209        0.095260     2.661426e+06
13  PLANT09              11        0.005014     1.400751e+05
14  PLANT03            1013        0.461714     1.289964e+07
15  PLANT13             490        0.223336     6.239707e+06
17  PLANT08              14        0.006381     1.782774e+05


In [43]:
plants = wh_subset["Plant ID"].tolist()
customers = demand["Customer"].tolist()

print(plants)
print(len(customers))

capacity = dict(zip(wh_subset["Plant ID"], wh_subset["scaled_capacity"]))
print(capacity)

['PLANT16', 'PLANT12', 'PLANT09', 'PLANT03', 'PLANT13', 'PLANT08']
42
{'PLANT16': 5819482.221103008, 'PLANT12': 2661426.2236554236, 'PLANT09': 140075.06440291702, 'PLANT03': 12899640.021832269, 'PLANT13': 6239707.414311758, 'PLANT08': 178277.3546946217}


In [44]:
demand_dict = dict(zip(demand["Customer"], demand["Unit quantity"]))
print(list(demand_dict.items())[:5])

[('V555555555555555555_17', 28995), ('V555555555555555555_42', 470632), ('V555555555555555555_45', 62745), ('V555555555555555555_46', 12080), ('V555555555555555_29', 102573)]


In [45]:
cost_dict = {}
for plant in plants:
    cost_dict[plant] = {}
    for customer in customers:
        match = plant_customer_cost[(plant_customer_cost["Plant Code"] == plant) & (plant_customer_cost["Customer"] == customer)]
        if not match.empty:
            cost_dict[plant][customer] = match["rate"].values[0]
print(cost_dict["PLANT03"])

{'V555555555555555555_17': np.float64(0.29304615384615385), 'V555555555555555555_42': np.float64(0.0484), 'V555555555555555555_45': np.float64(0.2376), 'V555555555555555555_46': np.float64(0.29960000000000003), 'V555555555555555_29': np.float64(0.04831928251121076), 'V555555555555555_44': np.float64(0.08), 'V55555555555555_8': np.float64(0.29621548387096774), 'V5555555555555_16': np.float64(0.23418181818181819), 'V55555555555_28': np.float64(0.04834059405940594), 'V555555555_14': np.float64(0.04835620437956204), 'V555555555_27': np.float64(0.048238771593090216), 'V555555555_3': np.float64(0.04818571428571428), 'V555555555_35': np.float64(0.0484), 'V55555555_0': np.float64(0.0484), 'V55555555_5': np.float64(0.0479503512880562), 'V55555555_7': np.float64(0.048316230366492144), 'V5555555_12': np.float64(0.0484), 'V5555555_19': np.float64(0.25951627906976743), 'V5555555_22': np.float64(0.20502049469964664), 'V5555555_30': np.float64(0.04806355140186916), 'V555555_24': np.float64(0.0484), '

In [48]:
from pulp import *
x = {}
for plant in plants:
    for customer in customers:
        if customer in cost_dict[plant]:
            x[(plant, customer)] = LpVariable(f"ship_{plant}_{customer}", lowBound = 0, cat = "Continuous")
print(len(x))

shortfall = {c : LpVariable(f"shortfall_{c}", lowBound=0) for c in customers}
BIG_PENALTY = 1000

prob = LpProblem("Plant_Customer_Optimization", LpMinimize)
prob += lpSum(cost_dict[p][c] * x[(p,c)] for (p,c) in x) +lpSum(BIG_PENALTY * shortfall[c] for c in customers)

for plant in plants:
    prob += lpSum(x[(plant, c)] for c in customers if (plant, c) in x) <= capacity[plant]

for customer in customers:
    prob += lpSum(x[(p, customer)] for p in plants if (p, customer) in x) + shortfall[customer] >= demand_dict[customer]

status = prob.solve()
print(LpStatus[status])

68
Optimal


In [49]:
print("total objective value: ", value(prob.objective))

actual_shipping_cost = sum(cost_dict[p][c] * x[(p,c)].varValue for (p,c) in x)
print("Actual shipping cost:", actual_shipping_cost)

total_shortfall = 0
for c in customers:
    if shortfall[c].varValue > 0:
        print(f"{c}: shortfall = {shortfall[c].varValue}")
        total_shortfall += shortfall[c].varValue
print("Total unmet demand:", total_shortfall)
print("As % of total demand:", (total_shortfall / sum(demand_dict.values())) * 100)

total objective value:  563057775.9127429
Actual shipping cost: 21142854.91274285
V555555555555555555_17: shortfall = 28995.0
V555555555555555555_45: shortfall = 21667.921
V555555555555555555_46: shortfall = 12080.0
V55555555555555_8: shortfall = 435751.0
V55555555_9: shortfall = 1414.0
V5555555_19: shortfall = 22394.0
V55555_10: shortfall = 2642.0
V5555_20: shortfall = 14247.0
V55_47: shortfall = 2724.0
Total unmet demand: 541914.921
As % of total demand: 2.2306127508505855


In [50]:
plant_totals = {}
for plant in plants:
    total = sum(x[(plant, c)].varValue for c in customers if (plant, c) in x)
    plant_totals[plant] = total
    print(f"{plant}: shipped {total} out of capacity {capacity[plant]}")

PLANT16: shipped 1633401.0 out of capacity 5819482.221103008
PLANT12: shipped 2661426.24 out of capacity 2661426.2236554236
PLANT09: shipped 140075.06 out of capacity 140075.06440291702
PLANT03: shipped 12899640.029 out of capacity 12899640.021832269
PLANT13: shipped 6239707.4 out of capacity 6239707.414311758
PLANT08: shipped 178277.35499999998 out of capacity 178277.3546946217
